# Mygrate Interactive Runner
This notebook sets up the environment and exposes a single interactive cell to invoke the Mygrate orchestrator (LLM-driven) or a deterministic pipeline.
- Imports and setup are done automatically.
- Provide a `project_path` and whether to use LLM.
- Visualizations are optional and will be generated if `migration_intelligence.json` is present.

In [ ]:
import os
import sys
import json
import subprocess
import importlib.util
from pathlib import Path
from IPython.display import display, Image, Markdown

repo_root = Path.cwd()
if not (repo_root / 'src').exists() and (repo_root.parent / 'src').exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


def check_javac():
    try:
        proc = subprocess.run(['javac', '-version'], capture_output=True, text=True, check=False)
        message = (proc.stderr or proc.stdout).strip()
        return proc.returncode == 0, message or 'javac available'
    except FileNotFoundError:
        return False, 'javac not found in PATH'


try:
    from src.workflow import app
    HAVE_WORKFLOW = True
    WORKFLOW_ERR = ''
except Exception as e:
    HAVE_WORKFLOW = False
    WORKFLOW_ERR = str(e)


def run_with_workflow(project_path: str, instruction: str, thread_id: str = 'mygrate_interactive'):
    if not HAVE_WORKFLOW:
        return False, f'workflow unavailable: {WORKFLOW_ERR}'

    config = {'configurable': {'thread_id': thread_id}}
    initial_state = {
        'project_path': project_path,
        'target_java_version': '17',
        'messages': [],
        'completed_tasks_summary': [],
        'dependencies': [],
        'compatibility_matrix': {},
        'migration_tasks': [],
        'current_instruction': instruction,
        'last_subagent_result': '',
        'next_node': 'supervisor',
    }

    try:
        app.invoke(initial_state, config=config)
        while True:
            state_val = app.get_state(config).values
            next_node = state_val.get('next_node', 'end')
            if next_node == 'end':
                break
            app.invoke(None, config=config)

        latest_state = app.get_state(config).values
        return True, latest_state.get('last_subagent_result', '')
    except Exception as e:
        return False, str(e)


def run_local_pipeline(project_path: str):
    script_path = repo_root / 'scripts' / 'run_local_pipeline.py'
    if not script_path.exists():
        return 1, {'error': 'run_local_pipeline.py not found'}

    spec = importlib.util.spec_from_file_location('run_local_pipeline', str(script_path))
    module = importlib.util.module_from_spec(spec)

    try:
        assert spec is not None and spec.loader is not None
        spec.loader.exec_module(module)
        rc = module.run(project_path)
    except Exception as e:
        return 1, {'error': str(e)}

    intel_path = repo_root / 'migration_intelligence.json'
    intel = {}
    try:
        intel = json.loads(intel_path.read_text(encoding='utf-8'))
    except Exception:
        pass
    return rc, intel


def visualize_if_present(intel_path: str = 'migration_intelligence.json', ask_user: bool = True):
    report_path = repo_root / intel_path
    if not report_path.exists():
        print(f'No intelligence file found at {report_path}')
        return

    if ask_user:
        answer = input('Generate visualizations now? [y/N]: ').strip().lower()
        if answer not in ('y', 'yes'):
            print('Skipping visualizations.')
            return

    try:
        from src.tools.visualization_engine import generate_dashboard, generate_cross_matrix
        generate_dashboard(str(report_path), 'dependency_graph.png')
        generate_cross_matrix(str(report_path), 'cross_matrix.png')
        display(Markdown('**Generated:** dependency_graph.png, cross_matrix.png'))
        if (repo_root / 'dependency_graph.png').exists():
            display(Image(filename=str(repo_root / 'dependency_graph.png')))
        if (repo_root / 'cross_matrix.png').exists():
            display(Image(filename=str(repo_root / 'cross_matrix.png')))
    except Exception as e:
        print('Visualization error:', e)


print('Setup complete. Use the next cell to run the pipeline interactively.')

Setup complete. Use the next cell to run the pipeline interactively.


In [ ]:
# Interactive runner cell
project = input('Project path (relative to repo root) [freshbrew_data/cantor]: ').strip() or 'freshbrew_data/cantor'
use_llm = input('Use LLM/orchestrator if available? [y/N]: ').strip().lower() in ('y', 'yes')
visualize = input('Prompt for visualizations after run? [Y/n]: ').strip().lower() not in ('n', 'no')
instruction = input('Instruction for agents [Quét dự án và sinh ma trận tương thích]: ').strip() or 'Quét dự án và sinh ma trận tương thích'

print('[+] Checking javac...')
javac_ok, javac_msg = check_javac()
print('javac:', javac_ok, javac_msg)

result = None
if use_llm and HAVE_WORKFLOW:
    print('[+] Attempting LLM-driven orchestrator (Supervisor)...')
    ok, payload = run_with_workflow(project, instruction)
    if ok:
        print('[+] Orchestrator completed. Agent output (last_subagent_result):')
        print(payload[:1000])
        result = payload
    else:
        print('[!] Orchestrator failed or returned no result. Falling back to deterministic pipeline.')
        rc, intel = run_local_pipeline(project)
        print(f'[+] Deterministic pipeline returned code {rc}')
        print(json.dumps(intel, indent=2)[:2000])
        result = intel
else:
    print('[+] Running deterministic pipeline (tools-only)')
    rc, intel = run_local_pipeline(project)
    print(f'[+] Deterministic pipeline returned code {rc}')
    print(json.dumps(intel, indent=2)[:2000])
    result = intel

if visualize:
    visualize_if_present('migration_intelligence.json', ask_user=True)

_mygrate_last_result = result
print('[+] Done. The variable `_mygrate_last_result` contains the returned object (or string).')

[+] Checking javac...
javac: True javac 17.0.12
[+] Attempting LLM-driven orchestrator (Supervisor)...
-> [SUPERVISOR] Phân tích Global State và yêu cầu từ User...
-> [SUPERVISOR] Lỗi Parse JSON. Trả luồng về end.
-> [SUPERVISOR] Quyết định gọi: end
[+] Orchestrator completed. Agent output (last_subagent_result):

